In [3]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

In [4]:
# 1. Tai du lieu goc
data_dir = Path('data')
df = pd.read_csv(data_dir / 'dataset.csv')
df_severity = pd.read_csv(data_dir / 'Symptom-severity.csv')

# 2. Chuan hoa ten trieu chung de so khop trong toan bo project.
def normalize_symptom_name(value):
    if not isinstance(value, str):
        return np.nan
    value = value.strip().lower()
    value = re.sub(r'\s*_\s*', '_', value)
    value = re.sub(r'\s+', '_', value)
    value = re.sub(r'_+', '_', value)
    return value.strip('_')

def clean_key(value):
    if not isinstance(value, str):
        return value
    return value.strip().lower().replace(' ', '').replace('_', '')

# 3. Tao tu dien trong so trieu chung.
severity_dict = {
    clean_key(row['Symptom']): int(row['weight'])
    for _, row in df_severity.iterrows()
}

# 4. Lay danh sach trieu chung duy nhat va dung ten cot da chuan hoa.
raw_symptom_cols = [col for col in df.columns if col != 'Disease']
all_symptoms = set()
for col in raw_symptom_cols:
    all_symptoms.update(normalize_symptom_name(s) for s in df[col].dropna())

sorted_symptoms = sorted(s for s in all_symptoms if isinstance(s, str))

# 5. Tao ma tran trong so. Moi hang la mot benh an da duoc ma hoa thanh vector.
X = pd.DataFrame(0, index=df.index, columns=sorted_symptoms)

for i, row in df.iterrows():
    row_symptoms = row[raw_symptom_cols].dropna().values
    for symptom in row_symptoms:
        symptom_name = normalize_symptom_name(symptom)
        if symptom_name in X.columns:
            X.at[i, symptom_name] = severity_dict.get(clean_key(symptom), 1)

# 6. Ma hoa ten benh va ghep voi ma tran trieu chung.
df['Disease_code'], _ = pd.factorize(df['Disease'])
final_df = pd.concat([df[['Disease', 'Disease_code']], X], axis=1)

# 7. Loai mau trung sau khi bien doi de train/test khong bi ro ri du lieu.
rows_before = len(final_df)
final_df = final_df.drop_duplicates(subset=['Disease'] + sorted_symptoms).reset_index(drop=True)
removed_rows = rows_before - len(final_df)

# 8. Luu vao thu muc data, khong luu cot index thua.
output_path = data_dir / 'transformed_dataset.csv'
final_df.to_csv(output_path, index=False)

print('Hoan thanh viec xu ly dataset')
print(f'Da loai {removed_rows} dong trung lap sau transform')
print(f'File da luu: {output_path}')
print(f'Kich thuoc du lieu sau transform: {final_df.shape[0]} dong x {final_df.shape[1]} cot')

Hoan thanh viec xu ly dataset
Da loai 4616 dong trung lap sau transform
File da luu: data\transformed_dataset.csv
Kich thuoc du lieu sau transform: 304 dong x 133 cot
